In [ ]:
# =============================================================
# ENGLISH -> HINDI Seq2Seq with BILSTM + BAHNDAU ATTENTION (NO CLASSES)
# Pure PyTorch implementation (no HuggingFace, no custom classes)
# =============================================================

import json
import random
import math
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from tqdm import tqdm

# -------------------------
# Config
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

DATA_JSON = "/kaggle/input/nlp-capstone-project-gru/train_data1.json"
SAVE_PATH = "bilstm_attention_model.pth"
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# -------------------------
# Load Data
# -------------------------
with open(DATA_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

pairs = [(v["source"], v["target"]) for v in data["English-Hindi"]["Train"].values()]
random.shuffle(pairs)
split_idx = int(0.9 * len(pairs))
train_pairs, val_pairs = pairs[:split_idx], pairs[split_idx:]
print("Train pairs:", len(train_pairs), "Val pairs:", len(val_pairs))

# -------------------------
# Tokenization + Vocab
# -------------------------
def tokenize(s):
    return s.strip().split()

PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"

src_vocab = {PAD, SOS, EOS, UNK}
tgt_vocab = {PAD, SOS, EOS, UNK}

for src, tgt in train_pairs:
    src_vocab.update(tokenize(src.lower()))
    tgt_vocab.update(tokenize(tgt.lower()))

src_vocab = sorted(list(src_vocab))
tgt_vocab = sorted(list(tgt_vocab))

src_w2i = {w: i for i, w in enumerate(src_vocab)}
tgt_w2i = {w: i for i, w in enumerate(tgt_vocab)}
src_i2w = {i: w for w, i in src_w2i.items()}
tgt_i2w = {i: w for w, i in tgt_w2i.items()}

SRC_VOCAB_SIZE, TGT_VOCAB_SIZE = len(src_vocab), len(tgt_vocab)
print("SRC vocab:", SRC_VOCAB_SIZE, "TGT vocab:", TGT_VOCAB_SIZE)

# -------------------------
# Dataloaders
# -------------------------
def sent_to_indices(sentence, w2i, add_sos_eos=False):
    tokens = tokenize(sentence.lower())
    inds = [w2i.get(t, w2i[UNK]) for t in tokens]
    if add_sos_eos:
        return [w2i[SOS]] + inds + [w2i[EOS]]
    return inds

def collate_fn(batch):
    src_seqs, tgt_seqs = [], []
    for src, tgt in batch:
        src_seqs.append(torch.tensor(sent_to_indices(src, src_w2i), dtype=torch.long))
        tgt_seqs.append(torch.tensor(sent_to_indices(tgt, tgt_w2i, add_sos_eos=True), dtype=torch.long))
    src_padded = pad_sequence(src_seqs, batch_first=True, padding_value=src_w2i[PAD])
    tgt_padded = pad_sequence(tgt_seqs, batch_first=True, padding_value=tgt_w2i[PAD])
    return src_padded.to(device), tgt_padded.to(device)

BATCH_SIZE = 32
train_loader = DataLoader(train_pairs, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_pairs, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# -------------------------
# Model Hyperparams
# -------------------------
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
ENC_HID = 256
DEC_HID = 512
ATT_HID = 256
DROPOUT = 0.2

# -------------------------
# Layers
# -------------------------
enc_embedding = nn.Embedding(SRC_VOCAB_SIZE, ENC_EMB_DIM, padding_idx=src_w2i[PAD]).to(device)
dec_embedding = nn.Embedding(TGT_VOCAB_SIZE, DEC_EMB_DIM, padding_idx=tgt_w2i[PAD]).to(device)

enc_lstm = nn.LSTM(ENC_EMB_DIM, ENC_HID, num_layers=1, bidirectional=True, batch_first=True).to(device)
dec_lstm = nn.LSTM(DEC_EMB_DIM + ENC_HID * 2, DEC_HID, batch_first=True).to(device)

enc2dec_h = nn.Linear(ENC_HID * 2, DEC_HID).to(device)
enc2dec_c = nn.Linear(ENC_HID * 2, DEC_HID).to(device)

attn_W = nn.Linear(DEC_HID + ENC_HID * 2, ATT_HID).to(device)
attn_v = nn.Linear(ATT_HID, 1, bias=False).to(device)

out_fc = nn.Linear(DEC_HID + ENC_HID * 2 + DEC_EMB_DIM, TGT_VOCAB_SIZE).to(device)
drop = nn.Dropout(DROPOUT)

params = list(enc_embedding.parameters()) + list(dec_embedding.parameters()) + \
         list(enc_lstm.parameters()) + list(dec_lstm.parameters()) + \
         list(enc2dec_h.parameters()) + list(enc2dec_c.parameters()) + \
         list(attn_W.parameters()) + list(attn_v.parameters()) + list(out_fc.parameters())

optimizer = optim.Adam(params, lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_w2i[PAD])

# -------------------------
# Helper Functions
# -------------------------
def make_src_mask(src):
    return (src != src_w2i[PAD]).to(device)

def encode_src(src):
    embedded = drop(enc_embedding(src))
    outputs, (h, c) = enc_lstm(embedded)  # outputs: [B, src_len, 2*ENC_HID]
    h_cat = torch.cat((h[-2], h[-1]), dim=1)
    c_cat = torch.cat((c[-2], c[-1]), dim=1)
    dec_h0 = torch.tanh(enc2dec_h(h_cat)).unsqueeze(0)
    dec_c0 = torch.tanh(enc2dec_c(c_cat)).unsqueeze(0)
    return outputs, (dec_h0, dec_c0)

def bahdanau_attention(dec_hidden, encoder_outputs, mask):
    batch_size, src_len, _ = encoder_outputs.size()
    dec_hidden = dec_hidden.unsqueeze(1).repeat(1, src_len, 1)
    energy = torch.tanh(attn_W(torch.cat((dec_hidden, encoder_outputs), dim=2)))
    attn_scores = attn_v(energy).squeeze(2)
    attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
    attn_weights = F.softmax(attn_scores, dim=1)
    context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
    return context, attn_weights

def decoder_step(input_token, prev_hidden, prev_cell, encoder_outputs, mask):
    emb = drop(dec_embedding(input_token)).unsqueeze(1)
    dec_hidden_last = prev_hidden[-1]
    context, _ = bahdanau_attention(dec_hidden_last, encoder_outputs, mask)
    rnn_input = torch.cat((emb, context.unsqueeze(1)), dim=2)
    output, (h, c) = dec_lstm(rnn_input, (prev_hidden, prev_cell))
    logits = out_fc(torch.cat((output.squeeze(1), context, emb.squeeze(1)), dim=1))
    return logits, h, c

# -------------------------
# Training / Validation
# -------------------------
def train_epoch(epoch, teacher_forcing_ratio=0.5):
    enc_lstm.train(); dec_lstm.train()
    total_loss = 0
    for src, tgt in tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        optimizer.zero_grad()
        src_mask = make_src_mask(src)
        enc_out, (h, c) = encode_src(src)
        input_token = tgt[:, 0]
        loss = 0
        for t in range(1, tgt.size(1)):
            logits, h, c = decoder_step(input_token, h, c, enc_out, src_mask)
            loss_t = criterion(logits, tgt[:, t])
            loss += loss_t
            teacher_force = random.random() < teacher_forcing_ratio
            input_token = tgt[:, t] if teacher_force else logits.argmax(1)
        loss = loss / (tgt.size(1) - 1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def eval_epoch():
    enc_lstm.eval(); dec_lstm.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in tqdm(val_loader, desc="Validation"):
            src_mask = make_src_mask(src)
            enc_out, (h, c) = encode_src(src)
            input_token = tgt[:, 0]
            loss = 0
            for t in range(1, tgt.size(1)):
                logits, h, c = decoder_step(input_token, h, c, enc_out, src_mask)
                loss += criterion(logits, tgt[:, t])
                input_token = logits.argmax(1)
            loss = loss / (tgt.size(1) - 1)
            total_loss += loss.item()
    return total_loss / len(val_loader)

# -------------------------
# Training Driver
# -------------------------
N_EPOCHS = 10
best_val = float("inf")
for epoch in range(1, N_EPOCHS + 1):
    tr_loss = train_epoch(epoch)
    val_loss = eval_epoch()
    print(f"Epoch {epoch}: Train {tr_loss:.4f} | Val {val_loss:.4f}")
    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            "enc_embedding": enc_embedding.state_dict(),
            "dec_embedding": dec_embedding.state_dict(),
            "enc_lstm": enc_lstm.state_dict(),
            "dec_lstm": dec_lstm.state_dict(),
            "enc2dec_h": enc2dec_h.state_dict(),
            "enc2dec_c": enc2dec_c.state_dict(),
            "attn_W": attn_W.state_dict(),
            "attn_v": attn_v.state_dict(),
            "out_fc": out_fc.state_dict(),
            "src_vocab": src_vocab,
            "tgt_vocab": tgt_vocab
        }, SAVE_PATH)
        print(f"✅ Saved improved model to {SAVE_PATH}")

# -------------------------
# Translate Function
# -------------------------
def translate(sentence, max_len=40):
    enc_lstm.eval(); dec_lstm.eval()
    tokens = sent_to_indices(sentence, src_w2i)
    src_tensor = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
    src_mask = make_src_mask(src_tensor)
    with torch.no_grad():
        enc_out, (h, c) = encode_src(src_tensor)
        input_token = torch.tensor([tgt_w2i[SOS]], dtype=torch.long).to(device)
        result = []
        for _ in range(max_len):
            logits, h, c = decoder_step(input_token, h, c, enc_out, src_mask)
            top1 = logits.argmax(1).item()
            if top1 == tgt_w2i[EOS]:
                break
            result.append(tgt_i2w[top1])
            input_token = torch.tensor([top1], dtype=torch.long).to(device)
    return " ".join(result)

# -------------------------
# Quick Test
# -------------------------
for s, t in random.sample(val_pairs, min(5, len(val_pairs))):
    pred = translate(s)
    print(f"\nEN: {s}\nGT: {t}\nPRED: {pred}")


Device: cuda
Train pairs: 72717 Val pairs: 8080
SRC vocab: 85283 TGT vocab: 88646


Validation: 100%|██████████| 253/253 [00:50<00:00,  4.96it/s]


Epoch 1: Train 6.4728 | Val 6.7196
✅ Saved improved model to bilstm_attention_model.pth


Validation: 100%|██████████| 253/253 [00:50<00:00,  4.97it/s]


Epoch 2: Train 5.3901 | Val 6.6455
✅ Saved improved model to bilstm_attention_model.pth


...

Validation: 100%|██████████| 253/253 [00:50<00:00,  4.99it/s]


Epoch 7: Train 3.4999 | Val 6.9402


Validation: 100%|██████████| 253/253 [00:50<00:00,  4.99it/s]


Epoch 8: Train 3.3696 | Val 6.9977


Epoch 9 [Train]:  31%|███       | 698/2273 [07:54<17:08,  1.53it/s]

In [2]:
import pandas as pd

VAL_FILE = "/kaggle/input/nlp-capstone-project-gru/val_data1.json"  # your validation JSON
OUTPUT_CSV = "/kaggle/working/Bi-LSTM-H.csv"

# Load validation data
with open(VAL_FILE, "r", encoding="utf-8") as f:
    val_data = json.load(f)

val_dict = val_data["English-Hindi"]["Validation"]

ids, sources, translations = [], [], []

print("Translating validation sentences...")
for k, v in tqdm(val_dict.items()):
    src = v["source"]
    pred = translate(src)
    ids.append(k)
    sources.append(src)
    translations.append(pred)

# Save to CSV
df = pd.DataFrame({"ID": ids, "Source": sources, "Translation": translations})
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\n✅ Saved translations to {OUTPUT_CSV}")


Translating validation sentences...


100%|██████████| 11543/11543 [07:49<00:00, 24.59it/s]


✅ Saved translations to /kaggle/working/Bi-LSTM-H.csv


In [ ]:
# =============================================================
# ENGLISH -> HINDI/BENGALI Seq2Seq with Bahdanau Attention using GRU + fastText
# =============================================================

import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from tqdm import tqdm
from gensim.models import KeyedVectors
from pathlib import Path

# -------------------------
# Configuration / device
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

DATA_JSON = "/kaggle/input/nlp-capstone-project-gru/train_data1.json"
SAVE_PATH = "/kaggle/working/seq2seq_gru_best.pth"
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# -------------------------
# 1) Load JSON -> pairs
# -------------------------
with open(DATA_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

train_dict = data["English-Hindi"]["Train"]
pairs = [(v["source"], v["target"]) for k, v in train_dict.items()]

random.shuffle(pairs)
split_idx = int(0.9 * len(pairs))
train_pairs = pairs[:split_idx]
val_pairs = pairs[split_idx:]

# -------------------------
# 2) Tokenization & Vocab
# -------------------------
def tokenize(s):
    return s.strip().split()

PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"

# Build vocab
def build_vocab(sentences):
    words = set()
    for s in sentences:
        for t in tokenize(s.lower()):
            words.add(t)
    vocab_list = [PAD, SOS, EOS, UNK] + sorted(words)
    w2i = {w:i for i,w in enumerate(vocab_list)}
    i2w = {i:w for w,i in w2i.items()}
    return vocab_list, w2i, i2w

src_vocab_list, src_w2i, src_i2w = build_vocab([s for s,t in train_pairs])
tgt_vocab_list, tgt_w2i, tgt_i2w = build_vocab([t for s,t in train_pairs])

SRC_VOCAB_SIZE = len(src_vocab_list)
TGT_VOCAB_SIZE = len(tgt_vocab_list)

# -------------------------
# 3) Load fastText embeddings
# -------------------------
# Download from https://fasttext.cc/docs/en/crawl-vectors.html
# English
ft_en_path = "/kaggle/input/fasttext-embeddings/cc.en.300.vec"
ft_en = KeyedVectors.load_word2vec_format(ft_en_path, binary=False)

# Hindi or Bengali (example for Hindi)
ft_tgt_path = "/kaggle/input/fasttext-embeddings/cc.hi.300.vec"
ft_tgt = KeyedVectors.load_word2vec_format(ft_tgt_path, binary=False)

EMB_DIM = 300  # fastText embedding size

# -------------------------
# 4) Initialize Embedding Matrices
# -------------------------
import numpy as np

def build_embedding_matrix(w2i, ft_model):
    vocab_size = len(w2i)
    emb_matrix = np.random.uniform(-0.1, 0.1, (vocab_size, EMB_DIM)).astype(np.float32)
    for word, idx in w2i.items():
        if word in ft_model:
            emb_matrix[idx] = ft_model[word]
    return torch.tensor(emb_matrix)

src_emb_matrix = build_embedding_matrix(src_w2i, ft_en)
tgt_emb_matrix = build_embedding_matrix(tgt_w2i, ft_tgt)

# -------------------------
# 5) DataLoader helpers
# -------------------------
def sent_to_indices(sentence, w2i, add_sos_eos=False):
    tokens = tokenize(sentence.lower())
    inds = [w2i.get(t, w2i[UNK]) for t in tokens]
    if add_sos_eos:
        return [w2i[SOS]] + inds + [w2i[EOS]]
    return inds

def collate_fn(batch):
    src_seqs, tgt_seqs = [], []
    for src, tgt in batch:
        sids = torch.tensor(sent_to_indices(src, src_w2i), dtype=torch.long)
        tids = torch.tensor(sent_to_indices(tgt, tgt_w2i, add_sos_eos=True), dtype=torch.long)
        src_seqs.append(sids)
        tgt_seqs.append(tids)
    src_padded = pad_sequence(src_seqs, batch_first=True, padding_value=src_w2i[PAD])
    tgt_padded = pad_sequence(tgt_seqs, batch_first=True, padding_value=tgt_w2i[PAD])
    return src_padded.to(device), tgt_padded.to(device)

BATCH_SIZE = 32
train_loader = DataLoader(train_pairs, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_pairs, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# -------------------------
# 6) Model components
# -------------------------
ENC_HID = 256
DEC_HID = 512
ATTN_DIM = 256
NUM_LAYERS = 1
DROPOUT = 0.2

# Embeddings
enc_embedding = nn.Embedding.from_pretrained(src_emb_matrix, freeze=False, padding_idx=src_w2i[PAD]).to(device)
dec_embedding = nn.Embedding.from_pretrained(tgt_emb_matrix, freeze=False, padding_idx=tgt_w2i[PAD]).to(device)

# Encoder GRU
enc_gru = nn.GRU(EMB_DIM, ENC_HID, num_layers=NUM_LAYERS, bidirectional=True, batch_first=True).to(device)

# Map encoder hidden to decoder init
enc2dec_h = nn.Linear(ENC_HID*2, DEC_HID).to(device)

# Attention
attn_W_enc = nn.Linear(ENC_HID*2, ATTN_DIM, bias=False).to(device)
attn_W_dec = nn.Linear(DEC_HID, ATTN_DIM, bias=False).to(device)
attn_v     = nn.Linear(ATTN_DIM, 1, bias=False).to(device)

# Decoder GRU
dec_gru = nn.GRU(EMB_DIM + ENC_HID*2, DEC_HID, num_layers=NUM_LAYERS, batch_first=True).to(device)

# Output projection
out_fc = nn.Linear(DEC_HID + ENC_HID*2 + EMB_DIM, TGT_VOCAB_SIZE).to(device)
drop = nn.Dropout(DROPOUT)

# -------------------------
# 7) Optimizer & Loss
# -------------------------
params = list(enc_gru.parameters()) + list(enc2dec_h.parameters()) + \
         list(attn_W_enc.parameters()) + list(attn_W_dec.parameters()) + list(attn_v.parameters()) + \
         list(dec_gru.parameters()) + list(out_fc.parameters())

optimizer = optim.Adam(params, lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_w2i[PAD])

# -------------------------
# 8) Source mask
# -------------------------
def make_src_mask(src_tensor):
    return (src_tensor != src_w2i[PAD]).to(device)

# -------------------------
# 9) Forward helpers
# -------------------------
def encode_src(src):
    embedded = drop(enc_embedding(src))
    outputs, h_n = enc_gru(embedded)
    h_n = h_n.view(NUM_LAYERS, 2, h_n.size(1), h_n.size(2))
    h_last = torch.cat((h_n[-1,0,:,:], h_n[-1,1,:,:]), dim=1)
    dec_h0 = torch.tanh(enc2dec_h(h_last)).unsqueeze(0)
    return outputs, dec_h0

def bahdanau_attention(dec_hidden, encoder_outputs, src_mask):
    enc_part = attn_W_enc(encoder_outputs)
    dec_part = attn_W_dec(dec_hidden).unsqueeze(1)
    energy = torch.tanh(enc_part + dec_part)
    scores = attn_v(energy).squeeze(2)
    scores = scores.masked_fill(~src_mask, -1e9)
    attn_weights = torch.softmax(scores, dim=1)
    context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
    return context, attn_weights

def decoder_step(input_token, prev_hidden, encoder_outputs, src_mask):
    emb = drop(dec_embedding(input_token)).unsqueeze(1)
    dec_hidden_last = prev_hidden[-1]
    context, attn_w = bahdanau_attention(dec_hidden_last, encoder_outputs, src_mask)
    rnn_input = torch.cat((emb, context.unsqueeze(1)), dim=2)
    output, h = dec_gru(rnn_input, prev_hidden)
    output = output.squeeze(1)
    logits = out_fc(torch.cat((output, context, emb.squeeze(1)), dim=1))
    return logits, h, attn_w

# -------------------------
# 10) Training / evaluation (same as your previous code)
# -------------------------
# ... (You can reuse train_one_epoch and evaluate_one_epoch functions here)
